# Predicting F1 Pit Stops — Optimal Pipeline (v4)
**Playground Series S6E5** | Target: `PitNextLap` | Metric: ROC-AUC

**Design choices** (from data audit):
- Train+test share all 104 races (row-level split) → `StratifiedKFold` matches the LB.
- `TyreLife` is present (raw); the comp removed `Normalized_TyreLife` — we reconstruct it from `Compound` quantiles.
- All test inputs are visible at inference, so neighbor features computed on `train+test` are legal.
- The **oracle feature** (`PitStop_next` when `lap_gap_next == 1`) is the strongest single signal.

**v3 → v4** (Tier 1 push toward 0.95):
1. **OOF target encoding** for high-card cats: `Driver`, `Race`, `Compound×Stint`, `Driver×Compound`, `Race×Compound`.
2. **Bagged CatBoost** — 3 seeds × varied hyperparams (depth/LR/l2), logit-rank averaged.
3. Final blend: logit-rank across {bagged CB, LGB, XGB}, auto-pick best by OOF AUC.

In [ ]:
import os, gc, warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import seaborn as sns
from scipy.stats import rankdata
from scipy.special import expit, logit
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance
import lightgbm as lgb
from catboost import CatBoostClassifier
import xgboost as xgb

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid', context='notebook')
SEED, N_FOLDS = 42, 5
DATA = Path('data')
OOF  = Path('oof'); OOF.mkdir(exist_ok=True)

## 1. Load

In [ ]:
train = pd.read_csv(DATA / 'train.csv')
test  = pd.read_csv(DATA / 'test.csv')
train.rename(columns={'LapTime (s)': 'LapTime'}, inplace=True)
test.rename(columns={'LapTime (s)': 'LapTime'}, inplace=True)
TARGET, ID = 'PitNextLap', 'id'
print(f'train: {train.shape}  test: {test.shape}  pos rate: {train[TARGET].mean():.4f}')

## 2. EDA
Five views, each chosen to surface a different driver of pit-stop probability.

In [ ]:
# --- 2a. Target rate by Compound ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
by_comp = train.groupby('Compound')[TARGET].agg(['mean','count']).sort_values('count', ascending=False)
by_comp['mean'].plot.bar(ax=axes[0], color='steelblue', edgecolor='k')
axes[0].set_title('Pit-next-lap rate by Compound'); axes[0].set_ylabel('P(pit)'); axes[0].set_xlabel('')
by_comp['count'].plot.bar(ax=axes[1], color='gray', edgecolor='k')
axes[1].set_title('Sample count by Compound'); axes[1].set_ylabel('rows'); axes[1].set_xlabel('')
plt.tight_layout(); plt.show()
by_comp

In [ ]:
# --- 2b. Pit hazard curves ---
fig, ax = plt.subplots(figsize=(10, 4.5))
for comp in ['SOFT','MEDIUM','HARD','INTERMEDIATE','WET']:
    s = train[train['Compound']==comp]
    if len(s) < 100: continue
    bins = np.arange(0, s['TyreLife'].max()+2, 2)
    s = s.assign(tl_bin=pd.cut(s['TyreLife'], bins))
    rate = s.groupby('tl_bin')[TARGET].mean()
    centers = [b.mid for b in rate.index]
    ax.plot(centers, rate.values, marker='o', label=comp, lw=2)
ax.set_xlabel('TyreLife (laps)'); ax.set_ylabel('P(pit next lap)')
ax.set_title('Pit-hazard curves'); ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# --- 2c. Compound × TyreLife heatmap ---
h = train.copy()
h['tl_bin'] = pd.cut(h['TyreLife'], bins=np.arange(0, 50, 3))
pivot = h.groupby(['Compound','tl_bin'])[TARGET].mean().unstack()
fig, ax = plt.subplots(figsize=(13, 3.5))
sns.heatmap(pivot, cmap='RdYlBu_r', cbar_kws={'label':'P(pit)'}, ax=ax)
ax.set_title('P(pit next lap) — Compound × TyreLife'); ax.set_xlabel('TyreLife bin'); ax.set_ylabel('')
plt.tight_layout(); plt.show()

In [ ]:
# --- 2d. Race-track avatars ---
def make_track_xy(seed, n=400):
    rng = np.random.default_rng(seed)
    theta = np.linspace(0, 2*np.pi, n, endpoint=False)
    r = np.ones_like(theta)
    for k in range(2, 8):
        amp = rng.uniform(-0.18, 0.18) / (k * 0.6)
        phase = rng.uniform(0, 2*np.pi)
        r = r + amp * np.cos(k*theta + phase)
    return r*np.cos(theta), r*np.sin(theta), theta

N_RP_BINS = 60
tmp = train.copy()
tmp['rp_bin'] = (tmp['RaceProgress'].clip(0, 1-1e-6) * N_RP_BINS).astype(int)
race_rate = tmp.groupby(['Race','rp_bin'])[TARGET].mean().unstack(fill_value=np.nan)
races = (race_rate.max(axis=1) - race_rate.min(axis=1)).sort_values(ascending=False).head(9).index.tolist()
fig, axes = plt.subplots(3, 3, figsize=(13, 13))
vmin, vmax = 0.0, float(np.nanpercentile(race_rate.values, 99))
for ax, race in zip(axes.flat, races):
    x, y_, theta = make_track_xy(seed=abs(hash(race)) % (2**32))
    bin_idx = ((theta / (2*np.pi)) * N_RP_BINS).astype(int)
    rates = race_rate.loc[race].reindex(range(N_RP_BINS)).values
    rate_per_pt = rates[bin_idx]
    pts = np.array([x, y_]).T.reshape(-1, 1, 2)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    lc = LineCollection(segs, cmap='RdYlBu_r', linewidth=10, norm=plt.Normalize(vmin, vmax))
    lc.set_array(rate_per_pt[:-1]); ax.add_collection(lc)
    ax.plot(x[0], y_[0], 'ks', markersize=7); ax.text(x[0]+0.05, y_[0]+0.05, 'S/F', fontsize=8)
    ax.set_xlim(-1.35, 1.35); ax.set_ylim(-1.35, 1.35); ax.set_aspect('equal'); ax.axis('off')
    n_rows = int((train['Race']==race).sum())
    pit_rate = train.loc[train['Race']==race, TARGET].mean()
    ax.set_title(f'{race}\nrows={n_rows:,}  pit rate={pit_rate:.3f}', fontsize=10)
fig.suptitle('Pit risk painted onto race-progress track avatars', fontsize=14, y=1.0)
fig.colorbar(lc, ax=axes, shrink=0.5, label='P(pit next lap)'); plt.show()

In [ ]:
# --- 2e. Stint length distribution per compound ---
stint_lens = train.groupby(['Race','Year','Driver','Stint','Compound']).size().reset_index(name='laps')
fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=stint_lens, x='Compound', y='laps',
            order=['SOFT','MEDIUM','HARD','INTERMEDIATE','WET'], ax=ax)
ax.set_title('Stint length by compound'); ax.set_ylabel('laps in stint')
plt.tight_layout(); plt.show()

## 3. Feature Engineering

In [ ]:
train['is_test'] = 0;  test['is_test'] = 1;  test[TARGET] = np.nan
full = pd.concat([train, test], ignore_index=True)
full = full.sort_values(['Race','Year','Driver','LapNumber']).reset_index(drop=True)
G    = ['Race','Year','Driver']
GS   = ['Race','Year','Driver','Stint']
RACE = ['Race','Year']

In [ ]:
g = full.groupby(G, sort=False)
for col in ['TyreLife','LapTime','Position','Stint','Compound',
            'PitStop','Cumulative_Degradation','LapTime_Delta',
            'Position_Change','LapNumber']:
    full[f'{col}_prev'] = g[col].shift(1)
    full[f'{col}_next'] = g[col].shift(-1)
for col in ['TyreLife','LapTime','Stint','Compound','PitStop','LapNumber']:
    full[f'{col}_prev2'] = g[col].shift(2)
    full[f'{col}_next2'] = g[col].shift(-2)

full['lap_gap_prev']  = full['LapNumber'] - full['LapNumber_prev']
full['lap_gap_next']  = full['LapNumber_next']  - full['LapNumber']
full['lap_gap_next2'] = full['LapNumber_next2'] - full['LapNumber']
full['lap_gap_prev2'] = full['LapNumber'] - full['LapNumber_prev2']

full['tyre_life_delta_next']  = full['TyreLife_next']  - full['TyreLife']
full['tyre_life_delta_next2'] = full['TyreLife_next2'] - full['TyreLife']
full['tyre_life_delta_prev']  = full['TyreLife']       - full['TyreLife_prev']
full['stint_change_next']     = (full['Stint_next']  > full['Stint']).astype('float')
full['stint_change_next2']    = (full['Stint_next2'] > full['Stint']).astype('float')
full['stint_change_prev']     = (full['Stint'] > full['Stint_prev']).astype('float')
full['compound_change_next']  = (full['Compound_next']  != full['Compound']).astype('float')
full['compound_change_next2'] = (full['Compound_next2'] != full['Compound']).astype('float')
full['just_pitted']           = (full['stint_change_prev'] == 1).astype('float')

full['oracle_pit_next'] = np.where(full['lap_gap_next'] == 1, full['PitStop_next'], np.nan)
full['oracle_gap2_pit'] = np.where(
    (full['lap_gap_next'] == 2) & (full['Stint_next'] > full['Stint']),
    1.0, np.nan)

full['laptime_diff_prev']  = full['LapTime'] - full['LapTime_prev']
full['laptime_diff_next']  = full['LapTime_next'] - full['LapTime']
full['laptime_diff_prev2'] = full['LapTime'] - full['LapTime_prev2']
full['pos_diff_prev']      = full['Position'] - full['Position_prev']
full['pos_diff_next']      = full['Position_next'] - full['Position']
full['laptime_roll3_mean'] = g['LapTime'].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
full['laptime_vs_recent']  = full['LapTime'] - full['laptime_roll3_mean']

gs = full.groupby(GS, sort=False)
full['stint_lap_min']      = gs['LapNumber'].transform('min')
full['stint_lap_max']      = gs['LapNumber'].transform('max')
full['stint_lap_count']    = gs['LapNumber'].transform('count')
full['stint_tyre_max']     = gs['TyreLife'].transform('max')
full['stint_progress']     = (full['LapNumber'] - full['stint_lap_min']) / (full['stint_lap_max'] - full['stint_lap_min'] + 1)
full['laps_left_in_stint'] = full['stint_lap_max'] - full['LapNumber']
full['tyre_left_in_stint'] = full['stint_tyre_max'] - full['TyreLife']
full['stint_laptime_mean'] = gs['LapTime'].transform('mean')
full['laptime_vs_stint']   = full['LapTime'] - full['stint_laptime_mean']
full['stint_deg_max']      = gs['Cumulative_Degradation'].transform('max')
full['deg_pct_of_max']     = full['Cumulative_Degradation'] / (full['stint_deg_max'] + 1e-6)
full['is_last_obs_of_stint'] = (full['LapNumber'] == full['stint_lap_max']).astype('float')

tr_mask = full['is_test'] == 0
cm = full[tr_mask].groupby('Compound')['TyreLife'].quantile(0.95).rename('compound_typical_max')
full = full.merge(cm, on='Compound', how='left')
full['tyre_life_normalised'] = full['TyreLife'] / (full['compound_typical_max'] + 1e-6)
full['tyre_life_over_max']   = (full['TyreLife'] > full['compound_typical_max']).astype('float')
csl = full[tr_mask].groupby('Compound')['stint_lap_count'].mean().rename('compound_avg_stint_len')
full = full.merge(csl, on='Compound', how='left')
full['stint_len_vs_compound_avg'] = full['stint_lap_count'] - full['compound_avg_stint_len']

rl = full.groupby(['Race','Year','LapNumber'])
full['n_drivers_this_lap']     = rl['Driver'].transform('count')
full['mean_position_this_lap'] = rl['Position'].transform('mean')
full['mean_laptime_this_lap']  = rl['LapTime'].transform('mean')
full['laptime_vs_field']       = full['LapTime'] - full['mean_laptime_this_lap']
full['field_pitrate_this_lap'] = rl['PitStop'].transform('mean')
rlc = full.groupby(['Race','Year','LapNumber','Compound'])
full['n_same_compound_this_lap'] = rlc['Driver'].transform('count')
full['frac_same_compound']       = full['n_same_compound_this_lap'] / full['n_drivers_this_lap']
ru = full.groupby(['Race','Year','LapNumber'])['PitStop'].sum().reset_index().rename(columns={'PitStop':'race_pits_this_lap'})
ru = ru.sort_values(['Race','Year','LapNumber'])
ru['race_pits_next_lap'] = ru.groupby(['Race','Year'])['race_pits_this_lap'].shift(-1)
full = full.merge(ru[['Race','Year','LapNumber','race_pits_next_lap']], on=['Race','Year','LapNumber'], how='left')

full['stints_done_so_far'] = full['Stint'] - 1
full['race_total_laps']    = full.groupby(RACE)['LapNumber'].transform('max')
full['laps_left_in_race']  = full['race_total_laps'] - full['LapNumber']
full['frac_laps_left']     = full['laps_left_in_race'] / (full['race_total_laps'] + 1e-6)
full['driver_pits_so_far'] = g['PitStop'].cumsum() - full['PitStop']

CAT_COLS = ['Driver','Compound','Race','Compound_prev','Compound_next','Compound_prev2','Compound_next2']
for c in CAT_COLS:
    full[c] = full[c].astype('category')

train_fe = full[full['is_test']==0].copy()
test_fe  = full[full['is_test']==1].copy()
print(f'train_fe: {train_fe.shape}  test_fe: {test_fe.shape}')

## 4. OOF target encoding
Bayesian-smoothed `(sum + α·μ) / (count + α)`. Computed per-fold on train so val rows never see their own target; test rows use all training data.

In [ ]:
TE_GROUPS = [
    ('Driver',),
    ('Race',),
    ('Compound','Stint'),
    ('Driver','Compound'),
    ('Race','Compound'),
    ('Driver','Race'),
]
ALPHA = 20.0
y_train_full = train_fe[TARGET].astype(int).values
global_mean = y_train_full.mean()
folds = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(train_fe, y_train_full))

def te_encode(cols, train_df, test_df, y, folds, alpha=ALPHA):
    cols = list(cols)
    key = 'te_' + '_x_'.join(cols)
    oof = np.full(len(train_df), global_mean, dtype=np.float32)
    for tr_idx, va_idx in folds:
        sub = train_df.iloc[tr_idx][cols].copy()
        sub['_y'] = y[tr_idx]
        agg = sub.groupby(cols, observed=True)['_y'].agg(['sum','count'])
        agg['te'] = (agg['sum'] + alpha * global_mean) / (agg['count'] + alpha)
        merged = (train_df.iloc[va_idx][cols]
                  .merge(agg.reset_index()[cols+['te']], on=cols, how='left'))
        oof[va_idx] = merged['te'].fillna(global_mean).astype(np.float32).values
    # test: full train
    sub = train_df[cols].copy(); sub['_y'] = y
    agg = sub.groupby(cols, observed=True)['_y'].agg(['sum','count'])
    agg['te'] = (agg['sum'] + alpha * global_mean) / (agg['count'] + alpha)
    merged = test_df[cols].merge(agg.reset_index()[cols+['te']], on=cols, how='left')
    te_pred = merged['te'].fillna(global_mean).astype(np.float32).values
    return key, oof, te_pred

for grp in TE_GROUPS:
    t0 = time.time()
    key, oof_te, pred_te = te_encode(grp, train_fe, test_fe, y_train_full, folds)
    train_fe[key] = oof_te
    test_fe[key]  = pred_te
    print(f'  {key}: mean={oof_te.mean():.4f}  std={oof_te.std():.4f}  {time.time()-t0:.1f}s')

drop_cols = [ID, TARGET, 'is_test']
feature_cols = [c for c in train_fe.columns if c not in drop_cols]
y    = y_train_full
X    = train_fe[feature_cols]
X_te = test_fe[feature_cols]
print(f'features: {len(feature_cols)}  train: {X.shape}  test: {X_te.shape}')

## 5. LightGBM

In [ ]:
lgb_params = dict(objective='binary', metric='auc',
                  learning_rate=0.03, num_leaves=511, min_child_samples=30,
                  feature_fraction=0.85, bagging_fraction=0.85, bagging_freq=1,
                  lambda_l2=2.0, verbose=-1, seed=SEED, n_jobs=-1)
oof_lgb  = np.zeros(len(X))
pred_lgb = np.zeros(len(X_te))
lgb_models = []
for f, (tr, va) in enumerate(folds):
    t0 = time.time()
    dtr = lgb.Dataset(X.iloc[tr], y[tr], categorical_feature=CAT_COLS)
    dva = lgb.Dataset(X.iloc[va], y[va], categorical_feature=CAT_COLS)
    m = lgb.train(lgb_params, dtr, 6000, valid_sets=[dva],
                  callbacks=[lgb.early_stopping(150), lgb.log_evaluation(0)])
    oof_lgb[va] = m.predict(X.iloc[va], num_iteration=m.best_iteration)
    pred_lgb   += m.predict(X_te, num_iteration=m.best_iteration) / N_FOLDS
    lgb_models.append(m)
    print(f'  fold {f+1}: AUC={roc_auc_score(y[va], oof_lgb[va]):.5f}  iter={m.best_iteration}  {time.time()-t0:.1f}s')
auc_lgb = roc_auc_score(y, oof_lgb)
np.savez(OOF/'lgb.npz', oof=oof_lgb, pred=pred_lgb)
print(f'LGB OOF AUC: {auc_lgb:.5f}')

## 6. Bagged CatBoost (3 seeds × hyperparam diversity)

In [ ]:
X_cb, X_te_cb = X.copy(), X_te.copy()
for c in CAT_COLS:
    X_cb[c]    = X_cb[c].astype(str).fillna('NA')
    X_te_cb[c] = X_te_cb[c].astype(str).fillna('NA')

CB_CONFIGS = [
    dict(seed=42,   depth=8, learning_rate=0.05, l2_leaf_reg=3.0, bagging_temperature=0.8, random_strength=1.0),
    dict(seed=7,    depth=7, learning_rate=0.06, l2_leaf_reg=2.0, bagging_temperature=1.0, random_strength=1.5),
    dict(seed=2023, depth=9, learning_rate=0.04, l2_leaf_reg=4.0, bagging_temperature=0.6, random_strength=0.8),
]

oof_cb_all  = []
pred_cb_all = []
for ci, cfg in enumerate(CB_CONFIGS):
    print(f'\nCB[{ci+1}/{len(CB_CONFIGS)}] cfg={cfg}')
    oof_i  = np.zeros(len(X))
    pred_i = np.zeros(len(X_te))
    for f, (tr, va) in enumerate(folds):
        t0 = time.time()
        m = CatBoostClassifier(
            iterations=4000, **{k:v for k,v in cfg.items() if k!='seed'},
            random_seed=cfg['seed'], eval_metric='AUC',
            cat_features=CAT_COLS, early_stopping_rounds=120,
            verbose=0, task_type='CPU')
        m.fit(X_cb.iloc[tr], y[tr], eval_set=(X_cb.iloc[va], y[va]), use_best_model=True)
        oof_i[va] = m.predict_proba(X_cb.iloc[va])[:,1]
        pred_i   += m.predict_proba(X_te_cb)[:,1] / N_FOLDS
        print(f'    fold {f+1}: AUC={roc_auc_score(y[va], oof_i[va]):.5f}  iter={m.tree_count_}  {time.time()-t0:.1f}s')
    auc_i = roc_auc_score(y, oof_i)
    np.savez(OOF/f'cb_{ci}.npz', oof=oof_i, pred=pred_i)
    oof_cb_all.append(oof_i)
    pred_cb_all.append(pred_i)
    print(f'  CB[{ci+1}] OOF AUC: {auc_i:.5f}')

# Bag CBs with logit-rank
def to_rank(x, eps=1e-6):
    r = rankdata(x) / len(x); return np.clip(r, eps, 1-eps)
def logit_rank(preds, weights):
    z = sum(w * logit(to_rank(p)) for w, p in zip(weights, preds))
    return expit(z / sum(weights))

auc_cb_each = [roc_auc_score(y, o) for o in oof_cb_all]
w_cb = np.array(auc_cb_each) ** 2
oof_cb  = logit_rank(oof_cb_all, w_cb)
pred_cb = logit_rank(pred_cb_all, w_cb)
auc_cb  = roc_auc_score(y, oof_cb)
np.savez(OOF/'cb_bag.npz', oof=oof_cb, pred=pred_cb)
print(f'\nCB individual OOF: {[f"{a:.5f}" for a in auc_cb_each]}')
print(f'CB BAGGED OOF AUC: {auc_cb:.5f}')

## 7. XGBoost (diversity)

In [ ]:
X_xgb, X_te_xgb = X.copy(), X_te.copy()
for c in CAT_COLS:
    X_xgb[c]    = X_xgb[c].astype('category')
    X_te_xgb[c] = X_te_xgb[c].astype('category')
xgb_params = dict(objective='binary:logistic', eval_metric='auc',
                  tree_method='hist', enable_categorical=True,
                  learning_rate=0.05, max_depth=8,
                  min_child_weight=20, subsample=0.85, colsample_bytree=0.85,
                  reg_lambda=2.0, seed=SEED, n_jobs=-1, verbosity=0)
oof_xgb  = np.zeros(len(X))
pred_xgb = np.zeros(len(X_te))
dte = xgb.DMatrix(X_te_xgb, enable_categorical=True)
for f, (tr, va) in enumerate(folds):
    t0 = time.time()
    dtr = xgb.DMatrix(X_xgb.iloc[tr], label=y[tr], enable_categorical=True)
    dva = xgb.DMatrix(X_xgb.iloc[va], label=y[va], enable_categorical=True)
    m = xgb.train(xgb_params, dtr, num_boost_round=4000,
                  evals=[(dva,'va')], early_stopping_rounds=120, verbose_eval=0)
    oof_xgb[va] = m.predict(dva, iteration_range=(0, m.best_iteration+1))
    pred_xgb   += m.predict(dte, iteration_range=(0, m.best_iteration+1)) / N_FOLDS
    print(f'  fold {f+1}: AUC={roc_auc_score(y[va], oof_xgb[va]):.5f}  iter={m.best_iteration}  {time.time()-t0:.1f}s')
auc_xgb = roc_auc_score(y, oof_xgb)
np.savez(OOF/'xgb.npz', oof=oof_xgb, pred=pred_xgb)
print(f'XGB OOF AUC: {auc_xgb:.5f}')

## 8. Diagnostics

In [ ]:
# Inter-model OOF correlation (bagged CB vs CB[0] vs CB[1] vs CB[2] vs LGB vs XGB)
oof_df = pd.DataFrame({
    'lgb':oof_lgb, 'cb_bag':oof_cb, 'cb_0':oof_cb_all[0], 'cb_1':oof_cb_all[1], 'cb_2':oof_cb_all[2], 'xgb':oof_xgb
})
fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(oof_df.corr(), annot=True, fmt='.5f', cmap='viridis', cbar=False, ax=ax)
ax.set_title('OOF prediction correlation')
plt.tight_layout(); plt.show()

In [ ]:
# Per-race AUC for bagged CB
race_auc = (train_fe.assign(oof=oof_cb)
            .groupby('Race')
            .apply(lambda d: roc_auc_score(d[TARGET], d['oof']) if d[TARGET].nunique()>1 else np.nan)
            .sort_values())
fig, ax = plt.subplots(figsize=(8, 8))
race_auc.head(20).plot.barh(ax=ax, color='steelblue')
ax.axvline(auc_cb, color='red', ls='--', label=f'overall {auc_cb:.4f}')
ax.set_title('20 weakest races (bagged CB OOF)'); ax.set_xlabel('AUC'); ax.legend()
plt.tight_layout(); plt.show()

## 9. Final logit-rank blend

In [ ]:
aucs = np.array([auc_lgb, auc_cb, auc_xgb])
w = aucs ** 2
oof_blend  = logit_rank([oof_lgb, oof_cb, oof_xgb], w)
pred_blend = logit_rank([pred_lgb, pred_cb, pred_xgb], w)
auc_blend  = roc_auc_score(y, oof_blend)

print(f'  LGB        OOF AUC: {auc_lgb:.5f}')
print(f'  CB BAGGED  OOF AUC: {auc_cb:.5f}')
print(f'  XGB        OOF AUC: {auc_xgb:.5f}')
print(f'  Logit-rank blend  : {auc_blend:.5f}')
candidates = [('lgb',auc_lgb,pred_lgb),('cb_bag',auc_cb,pred_cb),('xgb',auc_xgb,pred_xgb),
              ('blend',auc_blend,pred_blend)]
best = max(candidates, key=lambda t: t[1])
print(f'\nBest: {best[0]} @ {best[1]:.5f}')
final_pred = best[2]

## 10. Submission

In [ ]:
sub_out = test_fe[[ID]].copy()
sub_out[TARGET] = final_pred
sub_out.to_csv('submission.csv', index=False)
print('wrote submission.csv', sub_out.shape)
sub_out.head()